# Headless run: native-resolution LoFTR from 800 m against the base, per level and per height band (iteration 35)

Both runs: `Main_Pipeline_Headless`, 332 working photos of the new set, 6 standard queries + second pass, hint jittered 5 km, `LEVELS=[3..11]`, DINO on, copies off; the second run adds `--set NATIVE_FINE_FROM_HEIGHT_M=800` (LoFTR canvas two levels finer = 448 px, photo cleaned, tile locally normalised; coarse search unchanged). Points per photo of 10; the score out of 60 is 6 x the mean. `wrong` = answered more than 8 sigma off (-20).

Kill rule (written before the run): no wrong answer beyond 8 sigma in the native run, higher points above 800 m than the base, per-photo time inside the 22 s guard (the 30 s limit on the Jetson is about 3x slower than this server, so the server time has to stay under about 7 s to be safe).

In [1]:
import numpy as np, pandas as pd
R = "../results/native_fine/"
meta = pd.read_csv(R + "metadata_dev332.csv")[["filename", "alt"]]
runs = {"base": pd.read_csv(R + "details_base.csv"), "native from 800 m": pd.read_csv(R + "details_native800.csv")}
for name, d in runs.items():
    d = d.merge(meta, on="filename"); d["points"] = pd.to_numeric(d.points); d["answered"] = d.status == "match"; d["wrong"] = d.points < 0; runs[name] = d
    print(name, "photos", len(d), " score/60 %.2f" % (6 * d.points.mean()), " answered", int(d.answered.sum()), " wrong", int(d.wrong.sum()), " time mean %.2f s, max %.2f s" % (d.total_seconds.mean(), d.total_seconds.max()))

base photos 332  score/60 35.64  answered 247  wrong 0  time mean 2.20 s, max 9.39 s
native from 800 m photos 332  score/60 37.79  answered 259  wrong 0  time mean 4.25 s, max 17.90 s


In [2]:
def table(by, label, edges=None):
    rows = []
    for name, d in runs.items():
        key = pd.cut(d.alt, edges) if edges else d[by]
        for band, g in d.groupby(key, observed=True):
            rows.append(dict(run=name, band=str(band), n=len(g), points10=g.points.mean(), score60_share=6 * g.points.sum() / len(d), answered=g.answered.mean(), wrong=int(g.wrong.sum()),
                             seconds_mean=g.total_seconds.mean(), seconds_max=g.total_seconds.max(), native_share=(g.native_used.astype(str) == "True").mean()))
    t = pd.DataFrame(rows)
    wide = t.pivot(index="band", columns="run", values=["n", "points10", "answered", "wrong", "seconds_mean", "seconds_max"])
    print("\n==", label); print(wide.round(2).to_string())
    return t
by_height = table(None, "per height band (m)", [0, 300, 500, 800, 1000, 1300, 1600, 2100])
by_level = table("level", "per coarse level")


== per height band (m)
                 n                   points10                   answered                   wrong                   seconds_mean                   seconds_max                  
run           base native from 800 m     base native from 800 m     base native from 800 m  base native from 800 m         base native from 800 m        base native from 800 m
band                                                                                                                                                                           
(0, 300]      74.0              74.0     8.05              8.05     0.99              0.99   0.0               0.0         2.42              2.47        9.39              9.22
(1000, 1300]  43.0              43.0     4.12              5.28     0.58              0.70   0.0               0.0         2.35              6.18        6.93             17.90
(1300, 1600]  45.0              45.0     4.00              5.98     0.51              0.71   0.0

In [3]:
# which photos changed: gains and losses per photo, native against base
b = runs["base"].set_index("filename"); n = runs["native from 800 m"].set_index("filename")
j = b[["alt", "level", "points", "status"]].join(n[["points", "status", "total_seconds", "native_used"]], rsuffix="_native")
j["delta"] = j.points_native - j.points
print("photos better:", int((j.delta > 0).sum()), " worse:", int((j.delta < 0).sum()), " same:", int((j.delta == 0).sum()), " total delta points:", j.delta.sum(), " (score/60 change %.2f)" % (6 * j.delta.sum() / len(j) / 1))
print("status changes (base -> native):"); print(pd.crosstab(j.status, j.status_native))
print("worse photos:"); print(j[j.delta < 0][["alt", "level", "points", "points_native", "status", "status_native"]].round(0).to_string())

photos better: 24  worse: 9  same: 299  total delta points: 119  (score/60 change 2.15)
status changes (base -> native):
status_native  match  no_match
status                        
match            244         3
no_match          15        70
worse photos:
                                         alt  level  points  points_native status status_native
filename                                                                                       
1f2341b13a194cec875540476cf9a5ba.png   899.0      9      10              0  match      no_match
c2a97eeff6ae4032b40a17204360658f.png  1325.0     10      10              6  match         match
d5a19c9f1c9642a58ab54ca6db908e40.png  1231.0     10       6              3  match         match
a919824832f44814beed10dd22905571.png  1015.0     10       6              3  match         match
89db30d58cf44fbfa8d84368e442cb28.png  1441.0     11      10              6  match         match
5bc9589e591a44168de78230a2a38f31.png   906.0      9      10          